# DATA266 Lab 1 - Task 3: CycleGAN

This notebook is VSCode/Jupyter compatible. It uses only local Python paths and does not require cloud-specific packages or Drive mounting. Approved image data is in `task3_gan/data/`; checkpoints are saved locally.

In [ ]:
from pathlib import Path
import subprocess, sys
# Clone automatically when this notebook is opened directly from GitHub in Colab.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'task3_gan').is_dir() and (p / 'requirements.txt').exists()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path.cwd() / 'GenAI'
    if not (REPO_ROOT / 'task3_gan').is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/denishatank12/GenAI.git', str(REPO_ROOT)], check=True)
MEMBER_DIR = REPO_ROOT / 'task3_gan' / 'member_denisha'
DATA_ROOT = REPO_ROOT / 'task3_gan' / 'data'
CHECKPOINT_DIR = MEMBER_DIR / 'checkpoints_competition'
LOG_DIR = REPO_ROOT / 'reproducibility' / 'raw_logs'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True); LOG_DIR.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO_ROOT)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'torch', 'torchvision'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')], check=True)
import torch
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (GPU not available)')
for name in ['monet_jpg', 'photo_jpg']:
    folder = DATA_ROOT / name
    files = [p for p in folder.glob('**/*') if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'} and not p.name.startswith('._')] if folder.exists() else []
    print(f'{name}: {len(files)} images')
    assert files, f'Missing or empty approved-data folder: {folder}'

## Smoke test

The smoke test must pass before full training starts.

In [ ]:
subprocess.run([sys.executable, 'src/smoke_test.py', '--config', 'config.yaml'], cwd=MEMBER_DIR, check=True)

## Train and resume from local checkpoints

Run this cell again after an interruption; `--resume-latest` continues from the newest checkpoint.

In [ ]:
# Competition configuration: batch 4, 2,000 samples/epoch, 100 epochs,
# local checkpoints, smoke-gated training, and automatic official evaluation.
subprocess.run([sys.executable, 'run_task3.py', '--config', 'config_competition.yaml', '--checkpoint-dir', str(MEMBER_DIR / 'checkpoints_competition')], cwd=MEMBER_DIR, check=True)

In [ ]:
subprocess.run([sys.executable, 'src/plot_history.py'], cwd=MEMBER_DIR, check=True)
commands = [
 ['src/evaluate_metrics.py','--real','../data/monet_jpg','--fake','outputs/pred_B2A','--out','outputs/metrics_B2A.json'],
 ['src/evaluate_metrics.py','--real','../data/photo_jpg','--fake','outputs/pred_A2B','--out','outputs/metrics_A2B.json'],
 ['src/evaluate_submission.py','--real-monet','../data/monet_jpg','--real-photo','../data/photo_jpg','--gen-a2b','outputs/pred_A2B','--gen-b2a','outputs/pred_B2A','--out','outputs/submission.csv','--json-out','outputs/submission_metrics.json'],
 ['src/evaluate_lpips.py','--real','../data/monet_jpg','--recon','outputs/recon_A','--out','outputs/lpips_A.json'],
 ['src/evaluate_lpips.py','--real','../data/photo_jpg','--recon','outputs/recon_B','--out','outputs/lpips_B.json']
]
for command in commands: subprocess.run([sys.executable, *command], cwd=MEMBER_DIR, check=True)
print('Automated Task 3 evaluation complete. See task3_gan/member_denisha/outputs/.')

## Required final human audit

Two raters must complete the 30 fixed samples in `outputs/human_audit.csv`. Then run the final validation cell. Kaggle submission and leaderboard score remain manual.

In [ ]:
subprocess.run([sys.executable, 'src/audit_agreement.py'], cwd=MEMBER_DIR, check=True)
subprocess.run([sys.executable, 'reproducibility/validate_results.py', '--task', 'task3', '--root', str(REPO_ROOT)], cwd=REPO_ROOT, check=True)